In [1]:
# Run this cell first (only needed on Colab)
!pip install numpy matplotlib --quiet

# Lab 8: Mobile Robot Kinematics
## EE0849 Introduction to Robotics

In this lab you will implement the **unicycle**, **differential-drive**, and **bicycle** kinematic models, integrate trajectories under programmed commands, and compare how each model responds.

**Learning goals:**
1. Implement `q̇ = G(q) u` and integrate with simple Euler steps
2. Convert between differential-drive wheel speeds and `(v, ω)`
3. Implement the bicycle model
4. Compare the two models under identical commands

---
## Pre-Lab Answers

Fill in below before running anything:

**Q1.** Diff-drive, `r = 0.05 m`, `L = 0.25 m`, `φ̇_R = 10`, `φ̇_L = 4`:

- v = ?
- ω = ?
- R = ?

**Q2.** Car, `L = 2.5 m`, `v = 6 m/s`, `R = 30 m`. Steering angle δ = ?

**Q3.** Unicycle sketch for `(v, ω) = (1, 0.5)` over 2 s from `(0,0,0)`: *(attach picture or describe in one sentence)*

In [2]:
import numpy as np
import matplotlib.pyplot as plt

---
## Part 1 — Unicycle Integrator

In [3]:
def unicycle_step(q, u, dt):
    """Advance the unicycle by one Euler step.
    q = (x, y, theta) in meters/radians
    u = (v, omega) in m/s and rad/s
    dt = time step in seconds
    Returns the new q as a numpy array.
    """
    x, y, theta = q
    v, omega = u
    # TODO: compute x_dot, y_dot, theta_dot from the unicycle kinematic equations
    x_dot = None
    y_dot = None
    theta_dot = None
    return np.array([x + x_dot * dt, y + y_dot * dt, theta + theta_dot * dt])

In [4]:
def integrate_unicycle(q0, u_fn, T, dt=0.01):
    """Integrate the unicycle for T seconds.
    q0 is the initial state (x, y, theta).
    u_fn(t, q) returns the command (v, omega) at time t and state q.
    Returns an (N, 3) array of states sampled at dt.
    """
    N = int(T / dt) + 1
    traj = np.zeros((N, 3))
    traj[0] = q0
    for i in range(N - 1):
        t = i * dt
        u = u_fn(t, traj[i])
        traj[i + 1] = unicycle_step(traj[i], u, dt)
    return traj

In [ ]:
# TODO: integrate the three test cases from the handout:
#   a) straight:  (v, omega) = (1.0, 0.0)
#   b) spin:      (v, omega) = (0.0, 0.5)
#   c) arc:       (v, omega) = (1.0, 0.5)   -> expected radius 2 m
# Plot all three on the same axes.

q0 = np.array([0.0, 0.0, 0.0])
T = 4.0

traj_straight = integrate_unicycle(q0, lambda t, q: (1.0, 0.0), T)
traj_spin = integrate_unicycle(q0, lambda t, q: (0.0, 0.5), T)
traj_arc = integrate_unicycle(q0, lambda t, q: (1.0, 0.5), T)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(traj_straight[:, 0], traj_straight[:, 1], label='straight')
ax.plot(traj_spin[:, 0], traj_spin[:, 1], label='spin in place')
ax.plot(traj_arc[:, 0], traj_arc[:, 1], label='arc (R = 2 m)')
ax.scatter([0], [0], c='k', marker='x', label='start')
ax.set_aspect('equal')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Unicycle trajectories')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

---
## Part 2 — Differential Drive

In [ ]:
r_wheel = 0.04  # wheel radius (m)
L_track = 0.20  # track width (m)

def diff_drive_to_unicycle(phi_dot_R, phi_dot_L, r, L):
    """Return (v, omega) for given wheel speeds."""
    # TODO
    v = None
    omega = None
    return v, omega

def unicycle_to_diff_drive(v, omega, r, L):
    """Return (phi_dot_R, phi_dot_L) for desired body commands."""
    # TODO
    phi_dot_R = None
    phi_dot_L = None
    return phi_dot_R, phi_dot_L

# Sanity check: round-trip
v_target, omega_target = 0.3, 0.8
pR, pL = unicycle_to_diff_drive(v_target, omega_target, r_wheel, L_track)
v_back, omega_back = diff_drive_to_unicycle(pR, pL, r_wheel, L_track)
print(f'v: {v_target} -> {v_back}')
print(f'omega: {omega_target} -> {omega_back}')
assert np.isclose(v_target, v_back) and np.isclose(omega_target, omega_back)

### Step 2.2 — Drive a 2 m × 1 m rectangle

Drive the robot around a rectangle using piecewise-constant wheel commands. The easiest way is to specify the *body* command `(v, ω)` for each segment and compute how long to hold it so the distance or angle comes out right.

In [ ]:
# Each segment is (v, omega, duration)
v_f = 0.3       # m/s forward speed
omega_t = 0.5   # rad/s spin rate
quarter_turn = (np.pi / 2) / omega_t  # seconds for a 90 degree turn

segments = [
    (v_f, 0.0, 2.0 / v_f),   # forward 2 m
    (0.0, omega_t, quarter_turn),
    (v_f, 0.0, 1.0 / v_f),
    (0.0, omega_t, quarter_turn),
    (v_f, 0.0, 2.0 / v_f),
    (0.0, omega_t, quarter_turn),
    (v_f, 0.0, 1.0 / v_f),
]

# TODO: integrate the unicycle through the segment sequence and plot the path.
q = np.array([0.0, 0.0, 0.0])
dt = 0.01
path = [q.copy()]
for v, w, T_seg in segments:
    N = int(T_seg / dt)
    for _ in range(N):
        q = unicycle_step(q, (v, w), dt)
        path.append(q.copy())
path = np.array(path)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(path[:, 0], path[:, 1], lw=2)
ax.scatter(path[::20, 0], path[::20, 1], s=6, c='orange')
ax.set_aspect('equal')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Differential-drive rectangle')
ax.grid(alpha=0.3)
plt.show()

### Step 2.3 — Discussion

*Why can a differential-drive robot execute a spin-in-place step but a car-like robot cannot?*

**Your answer:** _(write 2–3 sentences here)_

---
## Part 3 — Bicycle Model

In [ ]:
def bicycle_step(q, u, L_wb, dt):
    """Advance the bicycle (car-like) model by one Euler step.
    q = (x, y, theta)
    u = (v, delta)      # rear-wheel speed, steering angle (radians)
    L_wb = wheelbase
    """
    x, y, theta = q
    v, delta = u
    # TODO: compute x_dot, y_dot, theta_dot for the bicycle model
    x_dot = None
    y_dot = None
    theta_dot = None
    return np.array([x + x_dot * dt, y + y_dot * dt, theta + theta_dot * dt])

In [ ]:
# Step 3.2 — Drive a circle and check the turning radius
L_wb = 2.5       # m
v_drive = 6.0    # m/s
delta = np.radians(10)
T = 15.0
dt = 0.01

q = np.array([0.0, 0.0, 0.0])
path = [q.copy()]
for _ in range(int(T / dt)):
    q = bicycle_step(q, (v_drive, delta), L_wb, dt)
    path.append(q.copy())
path = np.array(path)

R_expected = L_wb / np.tan(delta)
T_lap = 2 * np.pi * R_expected / v_drive
print(f'Expected turning radius: {R_expected:.2f} m')
print(f'Expected lap time:       {T_lap:.2f} s')

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(path[:, 0], path[:, 1])
ax.set_aspect('equal')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title(f'Bicycle circle: R = {R_expected:.1f} m')
ax.grid(alpha=0.3)
plt.show()

### Step 3.3 — Parallel parking preview

Try to reach `(0, 2, 0)` from `(0, 0, 0)` using only `(v, δ)`. Describe what happens and sketch the multi-step maneuver you would need:

**Your answer:** _(free-form)_

---
## Part 4 — Compare the Two Models

In [ ]:
# Step 4.1 — same commanded motion, different kinematic model.
# We will command v = 1 m/s for 5 s, then a turn for 3 s, then drive 2 s.
# Use a unicycle with omega = 0.4 rad/s and a bicycle with matching
# steering angle delta_eq computed from: omega = v * tan(delta) / L_wb.
L_wb = 1.0

def command(t):
    if t < 5.0:
        return 1.0, 0.0
    elif t < 8.0:
        return 1.0, 0.4      # for the unicycle
    else:
        return 1.0, 0.0

# TODO:
#   - compute delta_eq for the bicycle from omega = v*tan(delta)/L
#   - integrate both models over 10 s using the matching command
#   - plot the two trajectories on the same axes
delta_eq = np.arctan(0.4 * L_wb / 1.0)

q_u = np.array([0.0, 0.0, 0.0])
q_b = np.array([0.0, 0.0, 0.0])
path_u = [q_u.copy()]
path_b = [q_b.copy()]
dt = 0.01
T = 10.0
for i in range(int(T / dt)):
    t = i * dt
    v, omega = command(t)
    delta = delta_eq if omega != 0 else 0.0
    q_u = unicycle_step(q_u, (v, omega), dt)
    q_b = bicycle_step(q_b, (v, delta), L_wb, dt)
    path_u.append(q_u.copy())
    path_b.append(q_b.copy())
path_u = np.array(path_u)
path_b = np.array(path_b)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(path_u[:, 0], path_u[:, 1], label='unicycle', lw=2)
ax.plot(path_b[:, 0], path_b[:, 1], '--', label='bicycle', lw=2)
ax.set_aspect('equal')
ax.set_xlabel('x [m]')
ax.set_ylabel('y [m]')
ax.set_title('Unicycle vs bicycle under matched commands')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Step 4.2 — What happens at `v = 0`?

Set `v = 0` and command a non-zero turn. What happens to each model?

**Your answer:** _(1 sentence)_

### Step 4.3 — Minimum turning radius

With `L_wb = 1.0` m and `δ_max = 45°`, compute `R_min` for the bicycle. Compare with the unicycle's `R_min`. Which robot can reach `(0, 2, 0)` from `(0, 0, 0)` more easily?

In [ ]:
# TODO: compute and print the two minimum turning radii.
delta_max = np.radians(45)
L_wb_car = 1.0
R_min_bicycle = L_wb_car / np.tan(delta_max)
R_min_unicycle = 0.0
print(f'Bicycle  R_min = {R_min_bicycle:.3f} m')
print(f'Unicycle R_min = {R_min_unicycle:.3f} m')

---
## Wrap-up

Write 2–3 sentences about what surprised you or what you learned:

**Your answer:** _..._